In [2]:
# Cell 1 — Imports & setup
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from xgboost import XGBClassifier, Booster

import joblib
import sys
sys.path.append("..")

from src.transforms import (
    get_preprocessor_fixed,
    select_cols_indices,
    categorical_features,
)
from src.artifacts import save_joblib_with_manifest, save_json_with_manifest

SEED = 42
ART_ROOT = "../artifacts"
CAL_DIR = os.path.join(ART_ROOT, "calibration")
os.makedirs(CAL_DIR, exist_ok=True)


In [5]:
# Cell 2 — Load train & holdout
train_df = pd.read_csv("../artifacts/data/train.csv")
hold_df  = pd.read_csv("../artifacts/data/holdout.csv")

X_train = train_df.drop(columns=["num"])
y_train = train_df["num"].values

X_hold  = hold_df.drop(columns=["num"])
y_hold  = hold_df["num"].values

print("Train:", X_train.shape, "Hold:", X_hold.shape)


Train: (736, 15) Hold: (184, 15)


In [6]:
# Cell 3 — Feature selection info
featmap = json.load(open("../artifacts/feature_selection/feature_index_map.json"))

full_feature_names = featmap["full_feature_names"]
categories_list = featmap["categories_list"]

top_features = [
    'cp_asymptomatic', 'cp_atypical angina', 'sex_Female', 'thal_normal',
    'ca', 'chol', 'oldpeak', 'age', 'slope_flat', 'thal_reversable defect',
    'restecg_normal', 'cp_typical angina', 'cp_non-anginal',
    'thalch', 'restecg_st-t abnormality'
]

top_indices = [full_feature_names.index(f) for f in top_features]
print("Using k =", len(top_indices))


Using k = 15


In [8]:
# Cell 4 — Preprocessor + selector
selector = FunctionTransformer(
    func=select_cols_indices,
    kw_args={"indices": top_indices},
    validate=False
)

preprocessor = get_preprocessor_fixed(categories_list=categories_list)


In [9]:
# Cell 5 — Load trained booster
booster = Booster()
booster.load_model("../artifacts/tuning_optuna/xgb_k15_final.bst")

xgb_model = XGBClassifier()
xgb_model._Booster = booster
xgb_model._le = None
xgb_model.n_classes_ = 2


C:\Users\Nikhil\AppData\Local\Temp\ipykernel_16420\823962770.py:3: UserWarning: [10:29:05] WARNING: D:\bld\xgboost-split_1764148481205\work\src\c_api\c_api.cc:1511: Unknown file format: `bst`. Using UBJSON (`ubj`) as a guess.
  booster.load_model("../artifacts/tuning_optuna/xgb_k15_final.bst")


In [14]:
# Cell 6 — Uncalibrated evaluation (FIXED)

# Fit ONLY preprocessing + selector on TRAIN
pipe_uncal.fit(X_train, y_train)

# Now predict on holdout
y_proba_uncal = pipe_uncal.predict_proba(X_hold)[:, 1]

uncal_metrics = {
    "roc_auc": roc_auc_score(y_hold, y_proba_uncal),
    "avg_precision": average_precision_score(y_hold, y_proba_uncal),
    "brier": brier_score_loss(y_hold, y_proba_uncal)
}

print("Uncalibrated:", uncal_metrics)

save_json_with_manifest(
    uncal_metrics,
    os.path.join(CAL_DIR, "uncalibrated_metrics.json"),
    name="uncalibrated_metrics"
)


Uncalibrated: {'roc_auc': 0.8960425633668101, 'avg_precision': 0.8860054136289157, 'brier': 0.1295157937612339}


{'name': 'uncalibrated_metrics',
 'path': '../artifacts\\calibration\\uncalibrated_metrics.json',
 'sha256': 'bb85c998afcf7b0fa851a32951afd53fd710a6e3859f3b68a31e309f9839f351',
 'created_at': '2026-01-08 10:47:07',
 'notes': ''}

In [11]:
# Cell 7 — Sigmoid calibration
cal_sigmoid = CalibratedClassifierCV(
    estimator=pipe_uncal,
    method="sigmoid",
    cv=5
)

cal_sigmoid.fit(X_train, y_train)

y_proba_sig = cal_sigmoid.predict_proba(X_hold)[:, 1]

sig_metrics = {
    "roc_auc": roc_auc_score(y_hold, y_proba_sig),
    "avg_precision": average_precision_score(y_hold, y_proba_sig),
    "brier": brier_score_loss(y_hold, y_proba_sig)
}

print("Sigmoid:", sig_metrics)

save_joblib_with_manifest(
    cal_sigmoid,
    os.path.join(CAL_DIR, "xgb_k15_sigmoid.joblib"),
    name="xgb_k15_sigmoid"
)

save_json_with_manifest(
    sig_metrics,
    os.path.join(CAL_DIR, "sigmoid_metrics.json"),
    name="sigmoid_metrics"
)


Sigmoid: {'roc_auc': 0.9017216642754663, 'avg_precision': 0.9006401314772405, 'brier': 0.12416616456620137}


{'name': 'sigmoid_metrics',
 'path': '../artifacts\\calibration\\sigmoid_metrics.json',
 'sha256': '26a54e63ebd73f317923fff65ee4635528d4ecc0bc49ea1768174fe94cc7e058',
 'created_at': '2026-01-08 10:31:49',
 'notes': ''}

In [12]:
# Cell 8 — Isotonic calibration
cal_iso = CalibratedClassifierCV(
    estimator=pipe_uncal,
    method="isotonic",
    cv=5
)

cal_iso.fit(X_train, y_train)

y_proba_iso = cal_iso.predict_proba(X_hold)[:, 1]

iso_metrics = {
    "roc_auc": roc_auc_score(y_hold, y_proba_iso),
    "avg_precision": average_precision_score(y_hold, y_proba_iso),
    "brier": brier_score_loss(y_hold, y_proba_iso)
}

print("Isotonic:", iso_metrics)

save_joblib_with_manifest(
    cal_iso,
    os.path.join(CAL_DIR, "xgb_k15_isotonic.joblib"),
    name="xgb_k15_isotonic"
)

save_json_with_manifest(
    iso_metrics,
    os.path.join(CAL_DIR, "isotonic_metrics.json"),
    name="isotonic_metrics"
)


Isotonic: {'roc_auc': 0.9042922046867528, 'avg_precision': 0.8993812552310998, 'brier': 0.12276870367249279}


{'name': 'isotonic_metrics',
 'path': '../artifacts\\calibration\\isotonic_metrics.json',
 'sha256': '0e7a9eb7d0608f24174b0720a3dca6a19a971d040c4403dc9288353d7bd23990',
 'created_at': '2026-01-08 10:32:48',
 'notes': ''}

In [13]:
# Cell 9 — Calibration plots
plt.figure(figsize=(6,6))

for probs, label in [
    (y_proba_uncal, "Uncalibrated"),
    (y_proba_sig, "Sigmoid"),
    (y_proba_iso, "Isotonic")
]:
    frac_pos, mean_pred = calibration_curve(
        y_hold, probs, n_bins=10, strategy="uniform"
    )
    plt.plot(mean_pred, frac_pos, marker="o", label=label)

plt.plot([0,1], [0,1], "k--", label="Perfect calibration")
plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.title("Calibration Curves — XGBoost (k=15)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(CAL_DIR, "calibration_curve.png"), dpi=200)
plt.show()


NameError: name 'y_proba_uncal' is not defined

<Figure size 600x600 with 0 Axes>